<div style="font-family: system-ui, -apple-system, sans-serif; text-align: center; padding: 48px 24px 24px;">
    <div style="display: inline-block; background: #be0f05; color: white;
                padding: 12px 20px; border-radius: 12px; margin-bottom: 20px;">
        <span style="font-size: 36px; font-weight: 800; letter-spacing: -0.5px;">TIP4PATLIBS &ndash; Environment Setup &amp; Persistence</span>
    </div>
    <div style="font-size: 16px; color: #475569; margin-bottom: 8px; line-height: 1.6;">
        Understand the <strong>EPO TIP container</strong> &mdash; then make <strong>Claude&nbsp;Code, Git &amp; your training material</strong> survive every server restart.
    </div>
    <div style="font-size: 13px; color: #94a3b8; margin-bottom: 32px;">
        EPO Academy Training Material &nbsp;&middot;&nbsp; <a href="https://patentreports.depa.tech" target="_blank"
           style="color: #be0f05; text-decoration: none; font-weight: 600;">created by Arne Kr&uuml;ger</a>
    </div>
    <div style="background: #f8fafc; border-radius: 12px; padding: 24px 28px; max-width: 620px;
                margin: 0 auto; border: 1px solid #e2e8f0; text-align: left;">
        <div style="font-size: 14px; color: #334155; line-height: 1.9;">
            <strong>What this notebook covers</strong>
            <br/><em>Inspect the runtime</em>
            <br/>1 &nbsp;&middot;&nbsp; User identity &amp; home directory
            <br/>2 &nbsp;&middot;&nbsp; Filesystem mounts &mdash; what persists, what doesn't
            <br/>3 &nbsp;&middot;&nbsp; Training materials (Git worktree mount)
            <br/>4 &nbsp;&middot;&nbsp; Startup scripts &amp; dotfile persistence
            <br/>5 &nbsp;&middot;&nbsp; Python environment &amp; TIP library access
            <br/><br/><em>Set up Claude Code persistently</em>
            <br/>6 &nbsp;&middot;&nbsp; <strong>Install Claude&nbsp;Code</strong> (persistent npm prefix)
            <br/>7 &nbsp;&middot;&nbsp; <strong>Configure Claude for TIP</strong> (context + permissions)
            <br/>8 &nbsp;&middot;&nbsp; A status line for Claude&nbsp;Code
            <br/><br/><em>Git, storage &amp; material</em>
            <br/>9 &nbsp;&middot;&nbsp; <strong>Connect Git &amp; GitHub via SSH</strong>
            <br/>10 &nbsp;&middot;&nbsp; Disk usage overview
            <br/>11 &nbsp;&middot;&nbsp; <strong>Get the training material</strong> (clone the course repo)
        </div>
    </div>
    <div style="background: #fdf2f2; border-radius: 10px; padding: 16px 24px; max-width: 620px;
                margin: 28px auto 0; border: 1px solid #fecaca;">
        <div style="font-size: 14px; color: #404955; font-weight: 600;">&#9654; &nbsp;Run the cells top to bottom.</div>
        <div style="font-size: 12px; color: #64748b; margin-top: 6px; line-height: 1.6;">
            Only <code>/home/jovyan</code> survives a restart &mdash; everything else is rebuilt from the container image.
            Sections&nbsp;6&ndash;11 place Claude&nbsp;Code, its config, your Git identity, your SSH key and the course
            repo on that persistent volume, so you set them up <strong>once</strong> and they stay.
        </div>
    </div>
    <div style="margin-top: 20px; font-size: 12px; color: #cbd5e1;">
        Part of EPO TIP Working Group Sessions, 2026. &nbsp;Data: EPO PATSTAT Global, Autumn 2025.
    </div>
</div>

## 1. User identity & home directory

TIP uses `jovyan` (Jupyter convention) as the base user. A symlink maps your actual username to it.

In [ ]:
import os, subprocess

print(f"whoami:    {os.environ.get('USER', subprocess.getoutput('whoami'))}")
print(f"$HOME:     {os.environ['HOME']}")
print(f"realpath:  {os.path.realpath(os.environ['HOME'])}")
print()
# Show the symlink
!ls -la /home/ | grep -v '^\.\.'

## 2. Filesystem mounts

Only `/home/jovyan` is persistent (shown as `/` in JupyterLab's file browser).
Everything else — `/opt/conda`, the actual system root — is an **overlay** rebuilt from the container image on every restart.

In [ ]:
import pandas as pd
import subprocess

# Parse mount info for key paths
rows = []
for target in ["/home/jovyan", "/home/jovyan/training", "/home/jovyan/.cache", "/opt/conda", "/"]:
    result = subprocess.run(
        ["findmnt", "--target", target, "-n", "-o", "TARGET,SOURCE,FSTYPE"],
        capture_output=True, text=True
    )
    if result.stdout.strip():
        parts = result.stdout.strip().split(None, 2)
        source = parts[1] if len(parts) > 1 else "?"
        fstype = parts[2] if len(parts) > 2 else "?"
        # Shorten kubernetes paths for readability
        if "kubernetes.io" in source:
            source = "EmptyDir (Kubernetes)"
        elif source.startswith("/dev/"):
            source = source.split("[")[0]
        persistent = "Yes" if parts[0] == "/home/jovyan" and fstype != "overlay" else "No"
        if parts[0] == "/home/jovyan":
            persistent = "Yes"
        rows.append({"Mount": parts[0], "Source": source, "Type": fstype, "Persistent": persistent})

df = pd.DataFrame(rows)
df.style.set_caption("Container mount points")

## 3. Training materials (Git worktree mount)

The `~/training/` directory is mounted from a **Kubernetes EmptyDir** populated via a Git worktree checkout at pod startup. All files are owned by UID 65533, not your user. The directory is read-only.

In [ ]:
# Show the mount source (reveals the Git worktree path)
!findmnt --target ~/training -o SOURCE -n

print()

# Prove it's read-only in practice
import tempfile, os
test_path = os.path.expanduser("~/training/.write_test")
try:
    open(test_path, "w").close()
    os.remove(test_path)
    print("training/ is WRITABLE (unexpected!)")
except PermissionError:
    print("training/ is READ-ONLY (as expected)")

print()

# List top-level contents with ownership
!ls -la ~/training/

## 4. Startup scripts & dotfile persistence

The init script `/usr/bin/init_juser.sh` runs on **every container start** and overwrites
`.bashrc`, `.profile`, and `.condarc` with EPO defaults. Any customizations in those files are lost.

**Rule of thumb:** Put all shell customizations in `.bash_aliases` — it is never touched by init scripts.

In [ ]:
import os, datetime

# Check which dotfiles are overwritten on startup vs. persistent
dotfiles = [
    ".bashrc", ".profile", ".condarc",          # overwritten by init
    ".bash_aliases", ".gitconfig", ".npmrc",     # persistent
    ".ssh", ".claude", ".claude.json",           # persistent
]

# Container start time = .bashrc modification time (since it's always overwritten)
boot_time = os.path.getmtime(os.path.expanduser("~/.bashrc"))
boot_str = datetime.datetime.fromtimestamp(boot_time).strftime("%Y-%m-%d %H:%M:%S")
print(f"Container start time (from .bashrc): {boot_str}\n")

print(f"{'File':<20} {'Modified':<22} {'Overwritten on start?'}")
print("-" * 65)
for f in dotfiles:
    path = os.path.expanduser(f"~/{f}")
    if os.path.exists(path):
        mtime = os.path.getmtime(path)
        mtime_str = datetime.datetime.fromtimestamp(mtime).strftime("%Y-%m-%d %H:%M:%S")
        overwritten = "YES — lost on restart!" if abs(mtime - boot_time) < 60 else "No — persistent"
        print(f"{f:<20} {mtime_str:<22} {overwritten}")
    else:
        print(f"{f:<20} {'(not found)':<22}")

## 5. Python environment & TIP library access

The EPO TIP library (`epo.tipdata.patstat`) is installed in the base conda environment at `/opt/conda/...`.
When working inside a project venv, this library is invisible unless you explicitly bridge the gap.

In [ ]:
import sys

print(f"Python:       {sys.executable}")
print(f"sys.prefix:   {sys.prefix}")
print(f"base_prefix:  {sys.base_prefix}")
print(f"In a venv:    {sys.prefix != sys.base_prefix}")
print()

# Check if TIP library is available
try:
    import epo.tipdata.patstat as tip
    print(f"epo.tipdata.patstat: {tip.__file__}")
except ImportError as e:
    print(f"epo.tipdata.patstat: NOT AVAILABLE ({e})")
    print()
    print("Fix options:")
    print("  1. python -m venv --system-site-packages .venv")
    print('  2. echo "/opt/conda/lib/python3.12/site-packages" > .venv/lib/python3.12/site-packages/conda.pth')

## 6. Installing Claude Code persistently

`npm install -g` writes to `/opt/conda/` (overlay) by default — lost on restart.
The fix: redirect npm's global prefix to a persistent directory in your home.

Run the cell below for the **initial install** (one-time only, via npm — no Homebrew available on TIP).
After that, use `claude upgrade` to update.

In [ ]:
%%bash
set -e

# 1. Create persistent directory
mkdir -p ~/.npm-global

# 2. Set npm prefix (skip if already configured)
if ! grep -q 'npm-global' ~/.npmrc 2>/dev/null; then
    npm config set prefix ~/.npm-global
    echo "Set npm prefix to ~/.npm-global"
else
    echo "npm prefix already configured — skipping"
fi

# 3. Add to PATH in .bash_aliases (idempotent — skips if already present)
touch ~/.bash_aliases
if ! grep -q 'npm-global' ~/.bash_aliases 2>/dev/null; then
    printf '# Persistent npm global packages (survives container restarts)\nexport PATH="$HOME/.npm-global/bin:$PATH"\n' >> ~/.bash_aliases
    echo "Added PATH entry to ~/.bash_aliases"
else
    echo "PATH entry already in ~/.bash_aliases — skipping"
fi

# 4. Install Claude Code
export PATH="$HOME/.npm-global/bin:$PATH"
npm install -g @anthropic-ai/claude-code

# 5. Verify
echo ""
echo "Installed at: $(which claude)"
claude --version


## 7. Configure Claude for TIP

Two tweaks make Claude Code much more useful here, both stored under `~/.claude/` (persistent):

- **TIP context** → `~/.claude/CLAUDE.md`. Claude reads this automatically every session, so it already knows what survives a restart, how to connect to PATSTAT (`PatstatClient(env='PROD')`), and the `epo.tipdata` venv trap — you never have to re-explain your environment.
- **Generous permissions.** This is a throwaway virtual JupyterLab container, so we let Claude edit files and run shell commands without stopping to ask each time (`acceptEdits` + a broad tool allowlist). Saves a lot of clicking. Tighten this if you ever run Claude somewhere less disposable.

In [ ]:
import os, json

# ── 1. Give Claude the TIP environment context (persistent, user-global) ─────
# Written to ~/.claude/CLAUDE.md — Claude Code reads it automatically every
# session, so it "knows" TIP without you re-explaining it each time.
CLAUDE_MD = r"""# EPO TIP Environment — Claude Working Notes

This machine is an **EPO Technology Intelligence Platform (TIP)** JupyterLab
container. These facts change how you should work here — read before acting.

## Persistence (critical)
- **Only `/home/jovyan` survives a restart.** `~` is symlinked to it (a 30 GB
  persistent volume). Everything else — `/opt/conda`, `/`, all system dirs — is
  an **overlay rebuilt from the container image on every restart**; changes
  there are lost.
- The init script `/usr/bin/init_juser.sh` **overwrites `~/.bashrc`,
  `~/.profile`, `~/.condarc` on every start.** Never persist customizations
  there. Put shell/env tweaks in **`~/.bash_aliases`** (never touched), or in
  the owning tool's own config under `~`.
- Safe to write (persistent): `~/.bash_aliases`, `~/.gitconfig`, `~/.ssh/`,
  `~/.npm-global/`, `~/.config/`, `~/.claude/`, `~/.claude.json`.

## Tooling already set up here
- **Claude Code**: installed via a persistent npm prefix at `~/.npm-global`
  (PATH exported from `~/.bash_aliases`). Update with `claude upgrade` — do not
  `npm install -g` into `/opt/conda`, it won't survive a restart.
- **Git + GitHub over SSH**: identity in `~/.gitconfig`, key in
  `~/.ssh/id_ed25519`. Use SSH remotes (`git@github.com:org/repo.git`).
  Fine-grained PATs owned by a personal account *cannot* see an organization's
  private repos (GitHub returns 404) — don't suggest that route; SSH just works.

## PATSTAT / patent data
- The EPO data library `epo.tipdata.patstat` is installed **only in the base
  conda env** (`/opt/conda/lib/python3.12/site-packages`). Standard connection:
  ```python
  from epo.tipdata.patstat import PatstatClient
  import pandas as pd
  patstat = PatstatClient(env='PROD')                 # PROD = full production DB
  df = pd.DataFrame(patstat.sql_query(sql, use_legacy_sql=False))
  ```
- **Gotcha:** inside a `python -m venv` the library is invisible. Either create
  the venv with `--system-site-packages`, or add
  `/opt/conda/lib/python3.12/site-packages` via a `.pth` file. Simplest: use the
  base conda Python for PATSTAT work.
- Course/reference data is mounted **read-only** at `~/training/` (owned by UID
  65533; a Kubernetes EmptyDir populated from a git-worktree checkout). Never
  write there — copy out first if you need to modify.
- Data edition to assume: **PATSTAT Global, Autumn 2025** — but confirm at
  runtime, editions change.

## Python
- Base interpreter: `/opt/conda/bin/python` (Python 3.12, conda); pandas present.
- A notebook reporting `In a venv: False` on TIP is expected and fine.
"""
dst = os.path.expanduser("~/.claude/CLAUDE.md")
os.makedirs(os.path.dirname(dst), exist_ok=True)
with open(dst, "w") as f:
    f.write(CLAUDE_MD)
print(f"Wrote TIP context → {dst} ({len(CLAUDE_MD)} chars)")

# ── 2. Generous permissions — this is a throwaway virtual JupyterLab env ────
# Auto-accept edits and allow routine tools so Claude does not stop to ask for
# every file edit or shell command. Merges into settings.json (keeps theme etc).
sp = os.path.expanduser("~/.claude/settings.json")
s = json.load(open(sp)) if os.path.exists(sp) else {}
perms = s.setdefault("permissions", {})
perms["defaultMode"] = "acceptEdits"
allow = set(perms.get("allow", []))
allow.update(["Bash", "Edit", "Write", "MultiEdit", "NotebookEdit", "Read", "Grep", "Glob", "WebFetch"])
perms["allow"] = sorted(allow)
with open(sp, "w") as f:
    json.dump(s, f, indent=2)
print("Configured generous permissions (acceptEdits + broad tool allowlist).")
print("Restart Claude Code to pick up the changes.")


## 8. A status line for Claude Code

A one-line bar at the bottom of Claude Code showing **folder · git branch · context-window usage · session cost & time · model**. The script lives in `~/.claude/statusline-command.sh` and is registered in `~/.claude/settings.json`.

It uses the `jq` CLI to parse Claude's JSON input (present on TIP by default — the cell checks). It is registered under the `/home/jovyan/.claude/...` path because Claude Code runs as user `jovyan` on TIP. Restart Claude Code to see it.

In [ ]:
import os, json, shutil, stat, subprocess

# The status line shows folder · branch · context-usage bar · cost · model.
# It needs the jq CLI to parse Claude's JSON input (present on TIP by default).
print(f"jq: {shutil.which('jq') or 'NOT FOUND — the status line needs the jq CLI'}")

# ── 1. Write the script and make it executable ────────────────────────
STATUSLINE = r"""#!/usr/bin/env bash
# Claude Code status line script
# ~/.claude/statusline-command.sh

input=$(cat)

# Extract values
used=$(echo "$input" | jq -r '.context_window.used_percentage // empty')
model=$(echo "$input" | jq -r '.model.display_name // empty')
cwd=$(echo "$input" | jq -r '.workspace.current_dir // .cwd // empty')
cost=$(echo "$input" | jq -r '.cost.total_cost_usd // empty')
dur_ms=$(echo "$input" | jq -r '.cost.total_duration_ms // empty')

parts=''

# Folder: show basename of cwd, plus git branch in dim parens if inside a repo
if [ -n "$cwd" ]; then
  folder=$(basename "$cwd")
  parts=$'\033[34m'"${folder}"$'\033[0m'

  branch=$(git -C "$cwd" symbolic-ref --short -q HEAD 2>/dev/null \
           || git -C "$cwd" rev-parse --short HEAD 2>/dev/null)
  if [ -n "$branch" ]; then
    # Dirty indicator: yellow ✱ if there are uncommitted/untracked changes
    dirty=''
    if [ -n "$(git -C "$cwd" status --porcelain 2>/dev/null)" ]; then
      dirty=$' \033[33m✱\033[0m'
    fi
    parts="${parts} "$'\033[2m('"${branch}"$')\033[0m'"${dirty}"
  fi
fi

# Context window usage — visual progress bar
if [ -n "$used" ]; then
  used_int=${used%.*}
  if [ "$used_int" -ge 80 ]; then
    ctx_color=$'\033[31m'   # red — critical
  elif [ "$used_int" -ge 60 ]; then
    ctx_color=$'\033[33m'   # yellow — warning
  elif [ "$used_int" -ge 40 ]; then
    ctx_color=$'\033[32m'   # green — okay
  else
    ctx_color=$'\033[36m'   # cyan — fresh
  fi
  reset=$'\033[0m'
  dim=$'\033[2m'

  # Build a 10-block progress bar
  bar_width=10
  filled=$(( used_int * bar_width / 100 ))
  [ "$filled" -gt "$bar_width" ] && filled=$bar_width
  empty=$(( bar_width - filled ))

  bar=''
  for i in $(seq 1 "$filled"); do bar="${bar}█"; done
  for i in $(seq 1 "$empty");  do bar="${bar}░"; done

  ctx_part="${dim}ctx:${reset} ${ctx_color}[${bar}]${reset} ${ctx_color}${used_int}%${reset}"
  [ -n "$parts" ] && parts="${parts} "$'\033[2m|'$'\033[0m '
  parts="${parts}${ctx_part}"
fi

# Cost + session duration
if [ -n "$cost" ] || [ -n "$dur_ms" ]; then
  dim=$'\033[2m'
  reset=$'\033[0m'
  green=$'\033[32m'
  cd_part=''

  if [ -n "$cost" ]; then
    cost_fmt=$(printf '$%.2f' "$cost")
    cd_part="${green}${cost_fmt}${reset}"
  fi

  if [ -n "$dur_ms" ]; then
    total_s=$(( ${dur_ms%.*} / 1000 ))
    h=$(( total_s / 3600 ))
    m=$(( (total_s % 3600) / 60 ))
    s=$(( total_s % 60 ))
    if [ "$h" -gt 0 ]; then
      dur_fmt=$(printf '%dh%02dm' "$h" "$m")
    elif [ "$m" -gt 0 ]; then
      dur_fmt=$(printf '%dm%02ds' "$m" "$s")
    else
      dur_fmt=$(printf '%ds' "$s")
    fi
    [ -n "$cd_part" ] && cd_part="${cd_part} "
    cd_part="${cd_part}${dim}${dur_fmt}${reset}"
  fi

  [ -n "$parts" ] && parts="${parts} "$'\033[2m|'$'\033[0m '
  parts="${parts}${cd_part}"
fi

# Model name
if [ -n "$model" ]; then
  [ -n "$parts" ] && parts="${parts} "$'\033[2m|'$'\033[0m '
  parts="${parts}"$'\033[35m'"${model}"$'\033[0m'
fi

printf '%s' "$parts"
"""
dst = os.path.expanduser("~/.claude/statusline-command.sh")
os.makedirs(os.path.dirname(dst), exist_ok=True)
with open(dst, "w") as f:
    f.write(STATUSLINE)
os.chmod(dst, os.stat(dst).st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)
print(f"Wrote + chmod +x → {dst}")

# ── 2. Register it in settings.json using the resolved /home/jovyan path ────
# (Claude Code runs as user 'jovyan' on TIP; ~ resolves there via the symlink.)
script_real = os.path.realpath(dst)
sp = os.path.expanduser("~/.claude/settings.json")
s = json.load(open(sp)) if os.path.exists(sp) else {}
s["statusLine"] = {"type": "command", "command": f"bash {script_real}"}
with open(sp, "w") as f:
    json.dump(s, f, indent=2)
print(f"Registered statusLine → bash {script_real}")

# ── 3. Preview with a sample payload ─────────────────────────────
sample = ('{"context_window":{"used_percentage":42.5},'
          '"model":{"display_name":"Opus 4.8"},'
          '"workspace":{"current_dir":"' + os.getcwd() + '"},'
          '"cost":{"total_cost_usd":0.12,"total_duration_ms":95000}}')
out = subprocess.run(["bash", dst], input=sample, capture_output=True, text=True).stdout
print("\nPreview:", out)
print("Restart Claude Code to see the status line live.")


## 9. Connecting Git & GitHub persistently (SSH)

Git and `gh` are pre-installed but unconfigured. We authenticate with an **SSH key** rather than a token — it's simpler and far more robust: SSH authenticates by your **GitHub account**, so the key reaches every repository your account can access, including private organization repos you own or administer.

> ⚠️ **Why not a token?** A *fine-grained* Personal Access Token is scoped to a single **resource owner**. A token owned by your personal account can never see an **organization's** private repos, no matter how many boxes you tick — GitHub just returns *404 Not Found*. SSH sidesteps this entirely.

Everything needed persists under `/home/jovyan` and is **never overwritten** by the init script:

- **Identity** → `~/.gitconfig`
- **SSH key** → `~/.ssh/id_ed25519` (+ `.pub`)
- **Trusted host** → `~/.ssh/known_hosts`

So you do this **once** and it survives every container restart.

**How it works:** edit `GIT_NAME` / `GIT_EMAIL`, then run the cell. It sets your identity, generates a key if you don't have one, and tests the connection. On the **first run** it prints your **public key** and a link — add that key at [github.com/settings/ssh/new](https://github.com/settings/ssh/new), then **re-run the cell** to confirm. Copy the public key as a **single unbroken line** (it starts with `ssh-ed25519 `; a stray line break is the #1 cause of GitHub's *"key is invalid"*).

In [ ]:
import os, subprocess

# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  EDIT THESE — your commit identity                                         ║
# ╚═══════════════════════════════════════════════════════════════════════════╝
GIT_NAME  = "herrkrueger"                  # your GitHub username / display name
GIT_EMAIL = "arne.krueger@mtc.berlin"      # the email you commit as
# ─────────────────────────────────────────────────────────────────────────────

# ── 1. Git identity → ~/.gitconfig (persistent, never wiped by the init script)
for key, val in {"user.name": GIT_NAME, "user.email": GIT_EMAIL,
                 "init.defaultBranch": "main", "pull.rebase": "false"}.items():
    subprocess.run(["git", "config", "--global", key, val], check=True)
print(f"Git identity: {GIT_NAME} <{GIT_EMAIL}>  (~/.gitconfig)")

# ── 2. SSH key → ~/.ssh (persistent). Auth is by ACCOUNT, so this reaches every
#       repo your GitHub account can see, incl. private org repos you administer.
sshdir = os.path.expanduser("~/.ssh")
os.makedirs(sshdir, mode=0o700, exist_ok=True)
key = os.path.join(sshdir, "id_ed25519")
if not os.path.exists(key):
    subprocess.run(["ssh-keygen", "-t", "ed25519", "-C", GIT_EMAIL,
                    "-f", key, "-N", "", "-q"], check=True)
    print("Generated a new SSH key (~/.ssh/id_ed25519).")
else:
    print("SSH key already present (~/.ssh/id_ed25519).")

# ── 3. Trust github.com's host key (avoids the interactive prompt on first use)
known = os.path.join(sshdir, "known_hosts")
scan = subprocess.run(["ssh-keyscan", "-t", "ed25519,rsa", "github.com"],
                      capture_output=True, text=True)
seen = open(known).read() if os.path.exists(known) else ""
with open(known, "a") as f:
    for line in scan.stdout.splitlines():
        if line and line not in seen:
            f.write(line + "\n")

# ── 4. Test the connection ───────────────────────────────────────────────────
test = subprocess.run(["ssh", "-T", "-o", "StrictHostKeyChecking=accept-new",
                       "git@github.com"], capture_output=True, text=True)
msg = (test.stdout + test.stderr).strip()

if "successfully authenticated" in msg:
    print(f"\n✅ GitHub SSH works — {msg}")
else:
    print("\n⏳ GitHub does not know this key yet. One-time step:")
    print("   1. Open https://github.com/settings/ssh/new")
    print("   2. Paste the PUBLIC key below (a single unbroken line) and save")
    print("   3. Re-run this cell to verify\n")
    print("   ── your public key ──────────────────────────────────────────")
    print("   " + open(key + ".pub").read().strip())
    print("   ─────────────────────────────────────────────────────────────")


## 10. Disk usage overview

Check how much space you're using on your persistent volume vs the ephemeral overlay.

In [ ]:
!echo "=== Your persistent volume ===" && df -h /home/jovyan
!echo ""
!echo "=== Overlay (ephemeral) ===" && df -h /
!echo ""
!echo "=== Top directories by size in your home ===" && du -sh ~/*/  2>/dev/null | sort -rh | head -15

## 11. Getting the training material

The exercises for this course live in the **`mtcberlin/epo-tip4patlibs`** repository. The cell below clones it over **SSH** into `~/epo-tip4patlibs` on your persistent volume — or, if it's already there, pulls the latest changes.

Because it uses the SSH key from Section 7, it works for the private org repo without any token. Once cloned, the material appears in JupyterLab's file browser under `epo-tip4patlibs/`.

In [ ]:
import os, subprocess

# The training-material repository (edit if you forked it under your own account)
REPO = "mtcberlin/epo-tip4patlibs"

# Clone into your persistent home so it survives container restarts.
DEST = os.path.expanduser(f"~/{REPO.split('/')[1]}")   # ~/epo-tip4patlibs

if os.path.isdir(os.path.join(DEST, ".git")):
    print(f"Already cloned at {DEST} — pulling latest changes...")
    r = subprocess.run(["git", "-C", DEST, "pull", "--ff-only"],
                       capture_output=True, text=True)
    print((r.stdout or r.stderr).strip())
    if r.returncode != 0:
        print("(pull failed — resolve manually if you have local changes)")
else:
    print(f"Cloning {REPO} into {DEST} ...")
    # SSH URL → uses the key from Section 7, so private org repos work too.
    r = subprocess.run(["git", "clone", f"git@github.com:{REPO}.git", DEST],
                       capture_output=True, text=True)
    print((r.stdout or r.stderr).strip())
    if r.returncode != 0:
        raise RuntimeError(
            f"Clone failed. Did the SSH test in Section 7 succeed, and does your "
            f"account have access to {REPO}?\n{r.stderr}")
    print("Cloned.")

# Show what we got
if os.path.isdir(DEST):
    entries = sorted(e for e in os.listdir(DEST) if not e.startswith("."))
    print(f"\nContents of {DEST}:")
    for e in entries:
        tag = "/" if os.path.isdir(os.path.join(DEST, e)) else ""
        print(f"  {e}{tag}")
